# SSL-ECG Fixed Training on Google Colab

This notebook runs the FIXED SSL-ECG training code with proper subject-wise cross-validation.

**⚠️ IMPORTANT:**
- Free Colab has **12-hour timeout**
- For datasets with 450 ECGs, expect **8-15 hours** runtime
- **Save checkpoints frequently** to avoid losing progress
- Consider **Colab Pro** ($10/month) for 24-hour timeout

**Requirements:**
1. Upload your preprocessed data (.npy file)
2. Enable GPU: Runtime → Change runtime type → GPU
3. Keep browser tab open (or use Colab Pro)

**Author:** Fixed by Claude (2025-11-07)

---

## Step 1: Setup Environment

Install TensorFlow 1.14 and dependencies.

**Note:** This downgrades Python packages to work with TF 1.14.

In [ ]:
%%bash
# Check GPU availability
nvidia-smi

# Install TensorFlow 1.14 and dependencies
pip install tensorflow-gpu==1.14.0 --quiet
pip install tensorboard==1.14.0 --quiet
pip install scikit-learn==0.22.2 --quiet
pip install numpy==1.18.4 --quiet
pip install tqdm==4.36.1 --quiet
pip install pandas==0.25.1 --quiet
pip install mlxtend==0.17.0 --quiet
pip install scipy==1.4.1 --quiet
pip install opencv-python==4.2.0.34 --quiet

echo "✓ Installation complete!"

## Step 2: Clone Repository

Get the fixed SSL-ECG code.

In [ ]:
!git clone https://github.com/martinfrasch/SSL-ECGv2.git
%cd SSL-ECGv2
!git checkout claude/review-and-plan-improvements-011CUtwG7jQk8UkVsFwcjUs3
!ls -la

## Step 3: Upload Your Data

Upload your preprocessed ECG data file.

**Expected format:**
- File: `felicitys_mecg_0.npy`
- Shape: `(n_samples, 6 + signal_length)`
- Columns: `[subject_id, stress_label, pss, pdq, fsi, cortisol, ...ECG_windows...]`

In [ ]:
from google.colab import files
import os

# Create data directory
os.makedirs('data', exist_ok=True)

# Upload data file
print("Please upload your preprocessed data file (felicitys_mecg_0.npy):")
uploaded = files.upload()

# Move to data folder
for filename in uploaded.keys():
    !mv {filename} data/
    print(f"✓ Uploaded: {filename}")

# Verify
!ls -lh data/

## Step 4: Run Tests (Optional but Recommended)

Verify the fixes work correctly.

In [ ]:
!python tests/test_datasets.py

## Step 5: Configure Training

Set training parameters.

In [ ]:
# Training configuration
RANDOM_SEED = 42
EPOCHS = 30  # Reduce to 15 if worried about timeout
TOTAL_FOLDS = 5
SUBJECT_WISE = True  # ALWAYS True for correct experiments
NORMALIZE_FEATURES = True
DATA_TAG = 'mecg'

print(f"Configuration:")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Epochs: {EPOCHS}")
print(f"  Folds: {TOTAL_FOLDS}")
print(f"  Subject-wise CV: {SUBJECT_WISE} {'✓ CORRECT' if SUBJECT_WISE else '✗ INCORRECT'}")
print(f"  Feature normalization: {NORMALIZE_FEATURES}")
print(f"")
print(f"Estimated time: {EPOCHS * TOTAL_FOLDS * 2} - {EPOCHS * TOTAL_FOLDS * 3} minutes")
print(f"                = {(EPOCHS * TOTAL_FOLDS * 2) / 60:.1f} - {(EPOCHS * TOTAL_FOLDS * 3) / 60:.1f} hours")

## Step 6: Enable Keep-Alive (Important!)

This helps prevent Colab from disconnecting during long training.

In [ ]:
from IPython.display import display, Javascript

# Keep Colab alive
display(Javascript('''
function KeepClicking(){
console.log("Clicking");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(KeepClicking,60000)
'''))

print("✓ Keep-alive enabled (clicks connect button every 60 seconds)")

## Step 7: Run Training

**⚠️ IMPORTANT:**
- This will take 8-15 hours
- Keep browser tab open
- Checkpoints are saved every epoch
- If disconnected, you can resume from last checkpoint

In [ ]:
%cd codes

!python train_fixed.py \
    --random_seed {RANDOM_SEED} \
    --epochs {EPOCHS} \
    --total_folds {TOTAL_FOLDS} \
    --subject_wise {SUBJECT_WISE} \
    --normalize_features {NORMALIZE_FEATURES} \
    --data_tag {DATA_TAG}

## Step 8: Check Results

View the results after training completes.

In [ ]:
%cd ..

# List result files
!ls -lh output/ER_result/

# Show stress classification results
import pandas as pd
import glob

stress_files = glob.glob('output/ER_result/te_stress.csv')
if stress_files:
    df = pd.read_csv(stress_files[0])
    print("\nStress Classification Results:")
    print(df)
    print(f"\nMean Accuracy: {df['Accuracy'].mean():.3f} ± {df['Accuracy'].std():.3f}")
    print(f"Mean F1-Score: {df['F1'].mean():.3f} ± {df['F1'].std():.3f}")
    print(f"Mean ROC-AUC: {df['rocauc'].mean():.3f} ± {df['rocauc'].std():.3f}")
else:
    print("No results found yet.")

## Step 9: Download Results

Download all results to your local machine.

In [ ]:
# Zip results
!zip -r results.zip output/ models/

# Download
from google.colab import files
files.download('results.zip')

print("✓ Results downloaded!")

## Troubleshooting

### If Training Disconnects:

1. **Check if checkpoints exist:**
   ```python
   !ls -lh models/fold_0/
   ```

2. **Resume from checkpoint** (modify train_fixed.py to load checkpoint)

3. **Reduce epochs** to fit in 12 hours:
   - Try 15 epochs instead of 30
   - Or reduce to 3 folds instead of 5

### If Out of Memory:

1. **Reduce batch size** in train_fixed.py:
   ```python
   batchsize = 64  # Instead of 128
   ```

2. **Use smaller windows:**
   - Preprocess with 5-sec windows instead of 10-sec

### If Too Slow:

1. **Upgrade to Colab Pro** ($10/month)
   - 24-hour timeout
   - Better GPU (V100)
   - More RAM

2. **Or use GCP** (see setup_gcp_vm.sh script)

---

## Expected Performance

With the CORRECT subject-wise cross-validation:

- **Stress Accuracy:** 65-80% (not 85-95%!)
- **F1-Score:** 0.60-0.75
- **ROC-AUC:** 0.70-0.85

The ~20-30% drop from original paper is EXPECTED and CORRECT.
It represents true generalization to new patients.

---

**Questions?** See SETUP.md and CODE_REVIEW.md in the repository.